# Sites Enriched — Data Cleaning & Missing Value Handling

**Input :** `explo/andy/output/sites_enriched.csv` (151 sites × 36 columns)

**Output:** `explo/andy/output/sites_enriched_clean.csv`

| # | Issue | Column(s) | Strategy |
|---|---|---|---|
| 1 | Wrong dtype | `install_date` | Parse to datetime → extract `sensor_age_days` |
| 2 | 104 missing (68.9%) | `road_category_en` | Derive from `road_category_code` mapping |
| 3 | 32 missing (21.2%) | `road_name` | Fill with `'Unknown'` |
| 4 | 8 missing (5.3%) | `bike_lane_width_m` | Median imputation grouped by `morphology_en` |
| 5 | 3 missing (2.0%) | `district_code`, `road_code` | Fill with `'Unknown'` |
| 6 | 1 missing (0.7%) | `municipality` | Fill with `'Unknown'` |

## 1  Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

INPUT_PATH  = 'explo/andy/output/sites_enriched.csv'
OUTPUT_PATH = 'explo/andy/output/sites_enriched_clean.csv'

df = pd.read_csv(INPUT_PATH)
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

## 2  Overview — Missing Values

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
missing = df.isna().sum()
pct     = (missing / len(df) * 100).round(1)
dq = pd.DataFrame({'missing_n': missing, 'missing_%': pct, 'dtype': df.dtypes})
dq_nonzero = dq[dq['missing_n'] > 0].sort_values('missing_%', ascending=False)
print('Columns with missing values:')
display(dq_nonzero)

# ── Heatmap ───────────────────────────────────────────────────────────────────
cols_with_missing = dq_nonzero.index.tolist()
fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    df[cols_with_missing].isna().T,
    cmap=['#2ecc71', '#e74c3c'],
    cbar=False, ax=ax,
    linewidths=0.3, linecolor='white'
)
ax.set_title('Missing Value Map  (red = missing, green = present)', fontsize=13)
ax.set_xlabel('Site index')
plt.tight_layout()
plt.show()

## 3  Fix — `install_date` → datetime + `sensor_age_days`

In [ ]:
df['install_date'] = pd.to_datetime(df['install_date'], errors='coerce')

# Days since installation (reference: today)
REFERENCE_DATE = pd.Timestamp('today').normalize()
df['sensor_age_days'] = (REFERENCE_DATE - df['install_date']).dt.days

# Also extract install year in case it's useful as a categorical
df['install_year'] = df['install_date'].dt.year

print('install_date dtype :', df['install_date'].dtype)
print('sensor_age_days stats:')
print(df['sensor_age_days'].describe().round(0))
print()
print('install_year distribution:')
print(df['install_year'].value_counts().sort_index())

## 4  Fix — `road_category_en` (104 missing)

The `road_category_en` column is missing wherever `road_category_code` is one of:
`IW`, `OW`, `RW`, `EHW`, `VHW`. These are Wegenregister codes that were never
translated. We apply the official mapping.

In [ ]:
# Official Wegenregister road category code → English label
ROAD_CAT_MAP = {
    'EW'  : 'local_access_road',          # Erfontsluitingsweg
    'IW'  : 'local_road',                  # Interfaciale Weg
    'OW'  : 'minor_road',                  # Ondergeschikte Weg
    'RW'  : 'regional_road',               # Regionale Weg
    'EHW' : 'expressway_limited_access',   # Expressweg met beperkte toegankelijkheid
    'VHW' : 'high_capacity_road',          # Verbindingsweg met hoge capaciteit
    '-9'  : 'not_applicable',              # e.g. cycleways, footpaths
}

before = df['road_category_en'].isna().sum()
df['road_category_en'] = df['road_category_en'].fillna(
    df['road_category_code'].map(ROAD_CAT_MAP)
)
after = df['road_category_en'].isna().sum()

print(f'Missing before: {before}  →  after: {after}')
print()
print('road_category_en distribution:')
print(df['road_category_en'].value_counts())

## 5  Fix — `road_name` (32 missing)

In [ ]:
# Road name is not available for some segments (unnamed roads / cycleways)
# Filling with 'Unknown' — do NOT impute with another road name
print('Sites with missing road_name (by morphology):')
print(
    df[df['road_name'].isna()]['morphology_en']
    .value_counts()
    .rename('count')
)
print()

df['road_name'] = df['road_name'].fillna('Unknown')
print(f'road_name missing after fill: {df["road_name"].isna().sum()}')

## 6  Fix — `bike_lane_width_m` (8 missing)

These 8 sites have `bike_lane_source = 'missing:has_cycleway_no_width'` —
a cycleway is confirmed to exist but OSM has no numeric width tag.
Strategy: **median imputation grouped by `morphology_en`** (road type
is the strongest predictor of bike lane width).

In [ ]:
# Inspect missing rows
print('Missing bike_lane_width_m rows:')
display(df[df['bike_lane_width_m'].isna()][[
    'site_name', 'morphology_en', 'bike_lane_source', 'bike_lane_width_m'
]])

# Median width per road morphology type
print('\nMedian bike_lane_width_m by morphology_en:')
medians = df.groupby('morphology_en')['bike_lane_width_m'].median()
print(medians.round(2))

# Fill missing with group median, fall back to overall median
overall_median = df['bike_lane_width_m'].median()
df['bike_lane_width_m'] = df.groupby('morphology_en')['bike_lane_width_m'].transform(
    lambda x: x.fillna(x.median() if not np.isnan(x.median()) else overall_median)
)

# Flag imputed rows so ML models can use it as a feature
df['bike_width_imputed'] = df['bike_lane_source'] == 'missing:has_cycleway_no_width'

print(f'\nbike_lane_width_m missing after imputation: {df["bike_lane_width_m"].isna().sum()}')

## 7  Fix — Minor Missing Fields (district_code, road_code, municipality)

In [ ]:
# Inspect which sites have these missing
minor_cols = ['district_code', 'road_code', 'municipality']
print('Sites with minor missing fields:')
display(df[df[minor_cols].isna().any(axis=1)][['site_name', 'municipality', 'district_code', 'road_code']])

# Fill with 'Unknown'
for col in minor_cols:
    df[col] = df[col].fillna('Unknown')

print('\nMissing after fill:')
print(df[minor_cols].isna().sum())

## 8  Final Validation

In [ ]:
# ── Missing value check ───────────────────────────────────────────────────────
remaining = df.isna().sum()
remaining_nonzero = remaining[remaining > 0]

if remaining_nonzero.empty:
    print('✅  No missing values remaining.')
else:
    print('⚠️  Still missing:')
    print(remaining_nonzero)

print(f'\nFinal shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Columns added: sensor_age_days, install_year, bike_width_imputed')

# ── Distribution plots of key numeric features ────────────────────────────────
num_cols = [
    'bike_lane_width_m', 'dist_to_segment_m', 'length_m',
    'poi_shop_1000m', 'poi_education_1000m', 'sensor_age_days'
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, col in zip(axes, num_cols):
    ax.hist(df[col].dropna(), bins=20, color='#3498db', edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('')
    ax.set_ylabel('Count')
plt.suptitle('Distribution of Key Numeric Features (after cleaning)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── Categorical distributions ─────────────────────────────────────────────────
cat_cols = ['road_category_en', 'morphology_en', 'access_en']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    ax.barh(counts.index, counts.values, color='#2ecc71', edgecolor='white')
    ax.set_title(col, fontsize=11)
    ax.set_xlabel('Count')
plt.suptitle('Categorical Feature Distributions (after cleaning)', fontsize=13)
plt.tight_layout()
plt.show()

## 9  Export Cleaned Data

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f'✅  Saved: {OUTPUT_PATH}')
print(f'   Shape : {df.shape[0]} rows × {df.shape[1]} columns')
print()
print('Column summary:')
summary = pd.DataFrame({
    'dtype'  : df.dtypes,
    'missing': df.isna().sum(),
    'unique' : df.nunique(),
})
display(summary)